In [1]:
#----------------------------------
# Importar bibliotecas necessárias
#----------------------------------
import re
import pandas as pd
import pyreadr
from pathlib import Path

#----------------------------------
# DIRETORIO
#----------------------------------
base_dir = "C:/Users/gabri/OneDrive/Área de Trabalho/joao/TB/cesta-de-precos-pncp"

#----------------------------------
# INPUTS
#----------------------------------
DATASET_MEDICAMENTOS = "tasks/indicadores-MEL/inputs/medicamentos.rds"
DATASET_CONTRATACOES = "tasks/indicadores-MEL/inputs/contratacoes.rds"

#----------------------------------
# OUTPUTS
#----------------------------------


#----------------------------------
# Carrega o dataset de compras
#----------------------------------
DF_MEDICAMENTOS = pyreadr.read_r(f"{base_dir}/{DATASET_MEDICAMENTOS}")[None]
DF_CONTRATACOES = pyreadr.read_r(f"{base_dir}/{DATASET_CONTRATACOES}")[None]

### Inspeciona dataset `medicamentos` e filtra colunas necessarias para análise

In [2]:
DF_MEDICAMENTOS.head()

,path,numeroItem,endpoint,descricao,materialOuServico,materialOuServicoNome,valorUnitarioEstimado,valorTotal,quantidade,unidadeMedida,...,similaridade,medicamento,tipoMargemPreferencia,exigenciaConteudoNacional,Unnamed: 50,Unnamed: 51,tipoMargemPreferencia.codigo,tipoMargemPreferencia.nome,anomes_coleta,data.numeroControlePNCP
0,C:/Users/rdurl/OneDrive/Documentos/cesta-de-pr...,1,https://pncp.gov.br/api/pncp/v1/orgaos/8800090...,ÁCIDO VALPRÓICO 250 MG - COMPRIMIDO/CÁPSULA\r\...,M,Material,0.3,86625.0,288750.0,COMPRIMIDO (COM),...,0.7138707041740417,True,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-01,88000906000157-1-000376/2024
1,C:/Users/rdurl/OneDrive/Documentos/cesta-de-pr...,2,https://pncp.gov.br/api/pncp/v1/orgaos/8800090...,"ÁCIDO VALPRÓICO 500MG, COMPRIMIDO\r\n<p>Ácido ...",M,Material,0.51,210375.0,412500.0,COMPRIMIDO (COM),...,0.7305154204368591,True,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-01,88000906000157-1-000376/2024
2,C:/Users/rdurl/OneDrive/Documentos/cesta-de-pr...,3,https://pncp.gov.br/api/pncp/v1/orgaos/8800090...,ÁCIDO VALPRÓICO 50MG/ML XAROPE - FRASCO 100ML\...,M,Material,4.96,34720.0,7000.0,FRASCO (FRA),...,0.8097699880599976,True,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-01,88000906000157-1-000376/2024
3,C:/Users/rdurl/OneDrive/Documentos/cesta-de-pr...,4,https://pncp.gov.br/api/pncp/v1/orgaos/8800090...,AMITRIPTILINA CLORIDRATO 25MG - COMPRIMIDO\r\n...,M,Material,0.05,52500.0,1050000.0,COMPRIMIDO (COM),...,0.7163156270980835,True,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-01,88000906000157-1-000376/2024
4,C:/Users/rdurl/OneDrive/Documentos/cesta-de-pr...,5,https://pncp.gov.br/api/pncp/v1/orgaos/8800090...,"AMOXICILINA 500MG, COMPRIMIDO\r\n<p>Amoxicilin...",M,Material,0.24,25200.0,105000.0,COMPRIMIDO (COM),...,0.6775684356689453,True,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-01,88000906000157-1-000376/2024


In [3]:
colunas = [
    "data.numeroControlePNCP",
    "numeroItem",
    "tipoBeneficioNome",
    "aplicabilidadeMargemPreferenciaNormal",
    "percentualMargemPreferenciaNormal",
    "aplicabilidadeMargemPreferenciaAdicional",
    "percentualMargemPreferenciaAdicional",
    "tipoMargemPreferencia.codigo",
    "criterioJulgamentoNome",
    "tipoMargemPreferencia.nome",
    "tipoMargemPreferencia",
    "exigenciaConteudoNacional",
]

DF_MEDICAMENTOS_COLUNAS = DF_MEDICAMENTOS[colunas].copy()
print(f"Este dataset contém {DF_MEDICAMENTOS.shape[0]} linhas")
DF_MEDICAMENTOS_COLUNAS.head()

Este dataset contém 316661 linhas


,data.numeroControlePNCP,numeroItem,tipoBeneficioNome,aplicabilidadeMargemPreferenciaNormal,percentualMargemPreferenciaNormal,aplicabilidadeMargemPreferenciaAdicional,percentualMargemPreferenciaAdicional,tipoMargemPreferencia.codigo,criterioJulgamentoNome,tipoMargemPreferencia.nome,tipoMargemPreferencia,exigenciaConteudoNacional
0,88000906000157-1-000376/2024,1,Sem benefício,False,NaN,False,NaN,NaN,Menor preço,NaN,NaN,NaN
1,88000906000157-1-000376/2024,2,Sem benefício,False,NaN,False,NaN,NaN,Menor preço,NaN,NaN,NaN
2,88000906000157-1-000376/2024,3,Participação exclusiva para ME/EPP,False,NaN,False,NaN,NaN,Menor preço,NaN,NaN,NaN
3,88000906000157-1-000376/2024,4,Participação exclusiva para ME/EPP,False,NaN,False,NaN,NaN,Menor preço,NaN,NaN,NaN
4,88000906000157-1-000376/2024,5,Participação exclusiva para ME/EPP,False,NaN,False,NaN,NaN,Menor preço,NaN,NaN,NaN


### Inspeciona dataset `contratacoes` e filtra colunas necessarias

In [4]:
DF_CONTRATACOES.head()

,path,data.numeroControlePNCP,data.anoCompra,data.sequencialCompra,data.modalidadeId,data.modalidadeNome,data.modoDisputaId,data.modoDisputaNome,data.tipoInstrumentoConvocatorioCodigo,data.tipoInstrumentoConvocatorioNome,...,empty,endpoint,data.fontesOrcamentarias.codigo,data.fontesOrcamentarias.nome,data.fontesOrcamentarias.descricao,data.fontesOrcamentarias.dataInclusao,data.fontesOrcamentarias,data.orgaoEntidade,data.unidadeOrgao,data.amparoLegal
0,2025-01-Q1,88000906000157-1-000376/2024,2024,376,6,Pregão - Eletrônico,3,Aberto-Fechado,1,Edital,...,FALSE,https://pncp.gov.br/api/consulta/v1/contrataco...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-Q1,16726028000140-1-000043/2024,2024,43,6,Pregão - Eletrônico,1,Aberto,1,Edital,...,FALSE,https://pncp.gov.br/api/consulta/v1/contrataco...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-01-Q1,14814139000183-1-000207/2024,2024,207,6,Pregão - Eletrônico,3,Aberto-Fechado,1,Edital,...,FALSE,https://pncp.gov.br/api/consulta/v1/contrataco...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-01-Q1,01409580000138-1-001988/2024,2024,1988,6,Pregão - Eletrônico,1,Aberto,1,Edital,...,FALSE,https://pncp.gov.br/api/consulta/v1/contrataco...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-01-Q1,45787678000102-1-000494/2024,2024,494,6,Pregão - Eletrônico,1,Aberto,1,Edital,...,FALSE,https://pncp.gov.br/api/consulta/v1/contrataco...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
colunas = [
    "data.numeroControlePNCP",
    "data.srp",
]

DF_CONTRATACOES_COLUNAS = DF_CONTRATACOES[colunas].copy()
print(f"Este dataset contém {DF_CONTRATACOES.shape[0]} linhas")
DF_CONTRATACOES_COLUNAS.head()

Este dataset contém 79510 linhas


,data.numeroControlePNCP,data.srp
0,88000906000157-1-000376/2024,TRUE
1,16726028000140-1-000043/2024,TRUE
2,14814139000183-1-000207/2024,TRUE
3,01409580000138-1-001988/2024,TRUE
4,45787678000102-1-000494/2024,TRUE


### Unir `DF_CONTRATACOES` com `DF_MEDICAMENTOS` através da coluna `data.numeroControlePNCP`

In [ ]:
# Unir os dois dataframes pela coluna data.numeroControlePNCP
DF_UNIDO = DF_MEDICAMENTOS_COLUNAS.merge(
    DF_CONTRATACOES_COLUNAS,
    on="data.numeroControlePNCP",
    how="left",
)

# DF_JOIN_UNICO = (
#     DF_UNIDO
#     .groupby("data.numeroControlePNCP", as_index=False)
#     .agg(lambda x: list(x.dropna().unique()))
# )

print(f"Linhas após merge: {len(DF_UNIDO)}")

DF_UNIDO.head()

Linhas após merge: 491649


,data.numeroControlePNCP,numeroItem,tipoBeneficioNome,aplicabilidadeMargemPreferenciaNormal,percentualMargemPreferenciaNormal,aplicabilidadeMargemPreferenciaAdicional,percentualMargemPreferenciaAdicional,tipoMargemPreferencia.codigo,criterioJulgamentoNome,tipoMargemPreferencia.nome,tipoMargemPreferencia,exigenciaConteudoNacional,data.srp
0,88000906000157-1-000376/2024,1,Sem benefício,False,NaN,False,NaN,NaN,Menor preço,NaN,NaN,NaN,TRUE
1,88000906000157-1-000376/2024,2,Sem benefício,False,NaN,False,NaN,NaN,Menor preço,NaN,NaN,NaN,TRUE
2,88000906000157-1-000376/2024,3,Participação exclusiva para ME/EPP,False,NaN,False,NaN,NaN,Menor preço,NaN,NaN,NaN,TRUE
3,88000906000157-1-000376/2024,4,Participação exclusiva para ME/EPP,False,NaN,False,NaN,NaN,Menor preço,NaN,NaN,NaN,TRUE
4,88000906000157-1-000376/2024,5,Participação exclusiva para ME/EPP,False,NaN,False,NaN,NaN,Menor preço,NaN,NaN,NaN,TRUE


In [7]:
DF_UNIDO_BENEFICIO = DF_UNIDO[
    (DF_UNIDO["tipoBeneficioNome"] != "Sem benefício") 
    & (DF_UNIDO["tipoBeneficioNome"] != "Não se aplica")
]

print(f"Dataset após filtro: {len(DF_UNIDO_BENEFICIO)}")

DF_UNIDO_BENEFICIO.head()

Dataset após filtro: 138978


,data.numeroControlePNCP,numeroItem,tipoBeneficioNome,aplicabilidadeMargemPreferenciaNormal,percentualMargemPreferenciaNormal,aplicabilidadeMargemPreferenciaAdicional,percentualMargemPreferenciaAdicional,tipoMargemPreferencia.codigo,criterioJulgamentoNome,tipoMargemPreferencia.nome,tipoMargemPreferencia,exigenciaConteudoNacional,data.srp
2,88000906000157-1-000376/2024,3,Participação exclusiva para ME/EPP,False,NaN,False,NaN,NaN,Menor preço,NaN,NaN,NaN,TRUE
3,88000906000157-1-000376/2024,4,Participação exclusiva para ME/EPP,False,NaN,False,NaN,NaN,Menor preço,NaN,NaN,NaN,TRUE
4,88000906000157-1-000376/2024,5,Participação exclusiva para ME/EPP,False,NaN,False,NaN,NaN,Menor preço,NaN,NaN,NaN,TRUE
5,88000906000157-1-000376/2024,6,Participação exclusiva para ME/EPP,False,NaN,False,NaN,NaN,Menor preço,NaN,NaN,NaN,TRUE
6,88000906000157-1-000376/2024,7,Participação exclusiva para ME/EPP,False,NaN,False,NaN,NaN,Menor preço,NaN,NaN,NaN,TRUE


- Filtrar linhas onde `aplicabilidadeMargemPreferenciaNormal` = `TRUE`

In [8]:
# Seleciona linhas onde aplicabilidadeMargemPreferenciaNormal é True (cobre bools e strings 'true'/'TRUE')
DF_MARGEM_NORMAL = DF_UNIDO_BENEFICIO[DF_UNIDO_BENEFICIO["aplicabilidadeMargemPreferenciaNormal"].astype(str).str.lower() == "true"].copy()
print(f"Dataset após filtro: {len(DF_MARGEM_NORMAL)}")
DF_MARGEM_NORMAL.head()

Dataset após filtro: 1930


,data.numeroControlePNCP,numeroItem,tipoBeneficioNome,aplicabilidadeMargemPreferenciaNormal,percentualMargemPreferenciaNormal,aplicabilidadeMargemPreferenciaAdicional,percentualMargemPreferenciaAdicional,tipoMargemPreferencia.codigo,criterioJulgamentoNome,tipoMargemPreferencia.nome,tipoMargemPreferencia,exigenciaConteudoNacional,data.srp
594,00394544000185-1-002368/2024,1,Participação exclusiva para ME/EPP,True,5.0,True,10.0,NaN,Menor preço,NaN,NaN,NaN,TRUE
1453,00394544000185-1-002403/2024,5,Participação exclusiva para ME/EPP,True,5.0,True,10.0,NaN,Menor preço,NaN,NaN,NaN,TRUE
1458,00394544000185-1-002403/2024,10,Participação exclusiva para ME/EPP,True,5.0,True,10.0,NaN,Menor preço,NaN,NaN,NaN,TRUE
3245,00394544000185-1-002333/2024,3,Participação exclusiva para ME/EPP,True,10.0,False,NaN,NaN,Menor preço,NaN,NaN,NaN,TRUE
3246,00394544000185-1-002333/2024,3,Participação exclusiva para ME/EPP,True,10.0,False,NaN,NaN,Menor preço,NaN,NaN,NaN,TRUE
